# Simulación de Cohorte Sintética Poblacional

En este *notebook* vamos a generar y analizar un conjunto de datos sintético que representa a una población de **100.000 individuos**. El objetivo es modelar variables demográficas y de salud utilizando operaciones vectorizadas de `NumPy` y estructurando la información en un *DataFrame* de `Pandas`.

La generación de estos datos se basa en tres distribuciones estadísticas fundamentales para simular el comportamiento real de variables biológicas y epidemiológicas de forma independiente.

## Variables Simuladas

### 1. Datos de Identidad (Variables Categóricas)
*   **`Nombre` y `Apellido`:** Asignados aleatoriamente a partir de un listado base. Esta generación simula la repetición natural de nombres en una población utilizando muestreo con reemplazo.

### 2. Altura (Distribución Normal)
*   **Variable:** `Altura_cm` (Continua)
*   **Parámetros:** Media ($\mu$) = 165 cm | Desviación estándar ($\sigma$) = 12 cm.
*   **Justificación:** La altura humana es un rasgo poligénico. Al ser el resultado de la suma de múltiples factores genéticos y ambientales independientes, por el **Teorema del Límite Central**, los valores convergen naturalmente en una campana de Gauss.

### 3. Infecciones Respiratorias (Distribución de Poisson)
*   **Variable:** `Infecciones_Anuales` (Discreta)
*   **Parámetros:** Tasa de ocurrencia ($\lambda$) = 2 eventos por año.
*   **Justificación:** Modela el recuento de eventos aleatorios e independientes que ocurren en un intervalo de tiempo fijo. Es la distribución estándar en epidemiología para predecir cuántas veces una persona podría enfermarse en el lapso de un año.

### 4. Esquema de Vacunación (Distribución Binomial)
*   **Variable:** `Dosis_Vacuna` (Discreta)
*   **Parámetros:** Ensayos ($n$) = 3 dosis posibles | Probabilidad de éxito por dosis ($p$) = 0.85 (85%).
*   **Justificación:** Modela la cantidad de "éxitos" en una serie de ensayos de Bernoulli independientes (donde cada intento de vacunarse es un "Sí/No"). Resulta en un valor entero entre 0 y 3.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fijamos la semilla
#np.random.seed(42)
N = 10_000_000

# 1. Definimos las listas de posibles nombres y apellidos
nombres_base = ['Leandro', 'Victoria', 'Pablo', 'Lucía', 'Martín', 'Sofía', 'Alejandro', 'Camila', 'Juan', 'María']
apellidos_base = ['Ramírez', 'Gumina', 'Turjanski', 'Gómez', 'López', 'Rodríguez', 'Fernández', 'Pérez', 'González', 'Martínez']

# 2. Generamos 100.000 selecciones aleatorias de forma vectorizada
nombres = np.random.choice(nombres_base, size=N)
apellidos = np.random.choice(apellidos_base, size=N)

# 3. Variables estadísticas anteriores
altura = np.random.normal(loc=165.0, scale=12.0, size=N)
infecciones = np.random.poisson(lam=2.0, size=N)
dosis = np.random.binomial(n=3, p=0.85, size=N)

# 4. Consolidamos el DataFrame
df_poblacion = pd.DataFrame({
    'Nombre': nombres,
    'Apellido': apellidos,
    'Altura_cm': np.round(altura, 1), # Redondeamos a 1 decimal para mayor realismo
    'Infecciones_Anuales': infecciones,
    'Dosis_Vacuna': dosis
})




In [ ]:
# Ver los primeros registros
df_poblacion.head(10)

,Nombre,Apellido,Altura_cm,Infecciones_Anuales,Dosis_Vacuna
0,Alejandro,Fernández,162.5,6,3
1,Lucía,Ramírez,168.9,4,3
2,Camila,Gómez,168.3,1,2
3,Martín,Fernández,158.1,1,3
4,Alejandro,López,160.9,1,3
5,María,Turjanski,178.2,5,3
6,Pablo,González,149.5,3,3
7,Alejandro,Gumina,172.1,2,3
8,Camila,Gumina,178.6,1,3
9,Martín,Pérez,172.2,2,2


## Análisis estadístico de la población sintética

### Análisis de la Distribución Binomial: Población vs. Muestra

En la simulación anterior, generamos una variable llamada `Dosis_Vacuna` utilizando una distribución binomial. Esta variable representa la cantidad de dosis aplicadas a cada individuo, asumiendo un esquema completo de 3 dosis y una probabilidad de éxito (vacunación) constante.

En estadística, es fundamental distinguir entre los **parámetros** de la población completa (que son fijos y, en la vida real, usualmente desconocidos) y los **estadísticos** calculados a partir de una muestra pequeña (que utilizamos para estimar los poblacionales).

En el siguiente bloque de código, vamos a:
1. Calcular la **media poblacional ($\mu$)** y el **desvío estándar poblacional ($\sigma$)** utilizando a los 100.000 individuos.
2. Tomar tres muestras aleatorias pequeñas ($n=30$) y calcular sus respectivas **medias muestrales ($\bar{x}$)** y **desvíos muestrales ($s$)**.

In [ ]:
# Aislamos la variable de interés que actúa como nuestro universo de datos
poblacion_vacunas = df_poblacion['Dosis_Vacuna']

# 1. Parámetros Poblacionales (El valor "verdadero" y fijo)
mu_poblacional = poblacion_vacunas.mean()
sigma_poblacional = poblacion_vacunas.std(ddof=0) # ddof=0 porque es la población completa

print(f"--- PARÁMETROS POBLACIONALES (N={len(poblacion_vacunas)}) ---")
print(f"Media Poblacional (μ): {mu_poblacional:.4f} dosis")
print(f"Desvío Poblacional (σ): {sigma_poblacional:.4f} dosis\n")

# 2. Estadísticos Muestrales (El valor "estimado" con una muestra pequeña)
n_muestra = 10 # Definimos un tamaño de muestra chico

print(f"--- ESTADÍSTICOS MUESTRALES (n={n_muestra}) ---")
# Generamos 3 muestras aleatorias diferentes y comparamos
for i in range(1, 4):
    # random_state distinto asegura que agarremos personas distintas en cada iteración
    muestra = poblacion_vacunas.sample(n=n_muestra, random_state=42+i)

    media_muestral = muestra.mean()
    desvio_muestral = muestra.std(ddof=1) # ddof=1 es la corrección de Bessel para muestras

    print(f"Muestra {i} -> Media (x̄): {media_muestral:.4f} | Desvío (s): {desvio_muestral:.4f}")

--- PARÁMETROS POBLACIONALES (N=10000000) ---
Media Poblacional (μ): 2.5497 dosis
Desvío Poblacional (σ): 0.6186 dosis

--- ESTADÍSTICOS MUESTRALES (n=5) ---
Muestra 1 -> Media (x̄): 2.0000 | Desvío (s): 1.0000
Muestra 2 -> Media (x̄): 2.4000 | Desvío (s): 0.5477
Muestra 3 -> Media (x̄): 2.4000 | Desvío (s): 0.5477


> 💡 **Para pensar y debatir:**

Al ejecutar el código, notarás que la media poblacional ($\mu$) tiene un único valor fijo, pero las medias muestrales ($\bar{x}$) cambian cada vez que tomamos un grupo distinto de 30 personas.

1. **¿Por qué varía la media muestral en cada extracción?** Si todas las personas provienen de la misma población, ¿no debería darnos siempre el mismo promedio?
2. En un escenario real de investigación de salud pública, casi nunca tenemos el presupuesto ni el tiempo para encuestar a los 100.000 habitantes; solemos depender de **una sola muestra**. Si la media muestral varía al azar, ¿cómo podemos estar seguros de que nuestra única muestra representa fielmente la verdadera media poblacional? ¿Qué concepto estadístico nos ayuda a medir esa incertidumbre?
3. **El efecto del tamaño de la muestra ($n$):** Volvé a la celda de código anterior y modificá manualmente el valor de la variable `n_muestra`. Probá reemplazando el 30 por valores como $n=10$, $n=500$ y $n=5000$, volviendo a ejecutar la celda cada vez. ¿Cómo se comportan las medias muestrales ($\bar{x}$) al compararlas entre sí en cada caso? ¿Qué ocurre con la distancia entre la media muestral y la verdadera media poblacional a medida que el $n$ se va haciendo cada vez más grande?

> 🔍 **Un paso más allá: El Valor Teórico vs. Nuestra Población Simulada**

Por definición matemática, la media teórica (o Valor Esperado) de una distribución binomial es simplemente la cantidad de ensayos multiplicada por su probabilidad:
$$E[X] = n \times p$$
En nuestro esquema de vacunación ($n=3$ dosis, $p=0.85$), este valor teórico es exactamente **2.55**.

Sin embargo, al mirar la Media Poblacional ($\mu$) calculada por nuestro código, el resultado es *extremadamente cercano* a 2.55, pero **rara vez es idéntico**. ¿Por qué no da exacto?

Porque nuestra cohorte de 100.000 personas es grande, pero **finita**. La fórmula $n \times p$ representa la media de una población **infinita**. Al generar la cohorte intervino el azar computacional: es como tirar una moneda millones de veces; la proporción de caras se acerca a 50%, pero casi nunca es matemáticamente perfecta. Si tuviéramos memoria infinita y pudiéramos generar una población de tamaño $N \to \infty$, nuestra media poblacional calculada coincidiría exactamente con la Esperanza Matemática.

In [ ]:
# --- Visualización ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.set_theme(style="whitegrid")

# Histograma Normal
sns.histplot(df_poblacion['Altura_cm'], bins=50, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Normal: Altura (cm)')

# Gráfico de barras Poisson
sns.countplot(data=df_poblacion, x='Infecciones_Anuales', ax=axes[1], color='lightgreen')
axes[1].set_title('Poisson: Infecciones por Año')

# Gráfico de barras Binomial
sns.countplot(data=df_poblacion, x='Dosis_Vacuna', ax=axes[2], color='salmon')
axes[2].set_title('Binomial: Dosis de Vacuna (n=3)')

plt.tight_layout()
plt.show()

# Ver los primeros registros
print(df_poblacion.head())

### Análisis de la Distribución de Poisson: Infecciones Anuales

Ahora vamos a analizar la variable `Infecciones_Anuales`, que generamos usando una distribución de Poisson. Esta distribución es ideal para modelar conteos de eventos (como enfermedades, accidentes o llegadas de pacientes a una guardia) que ocurren en un intervalo de tiempo fijo.

Cuando creamos nuestra población sintética, definimos una tasa promedio de ocurrencia ($\lambda$, la letra griega "lambda") igual a 2 infecciones por año.

Una propiedad matemática fascinante de la distribución de Poisson es que **su media teórica y su varianza teórica son exactamente iguales** al parámetro $\lambda$. Por lo tanto, el desvío estándar teórico es la raíz cuadrada de $\lambda$ ($\sqrt{\lambda}$).

En el siguiente bloque de código vamos a comparar:
1. El cálculo matemático **teórico**.
2. Los parámetros reales de nuestra **población finita** de 100.000 habitantes.
3. Los estadísticos obtenidos al sacar una **muestra aleatoria** pequeña ($n=50$).

In [ ]:
import math

# Aislamos la variable de infecciones
poblacion_infecciones = df_poblacion['Infecciones_Anuales']

# 1. Valores Teóricos (Matemáticos)
lam = 2.0
media_teorica = lam                # acá repetimos lam, pero conviene guardarlo en una variable global
desvio_teorico = math.sqrt(lam)

print("--- 1. VALORES TEÓRICOS (Población infinita) ---")
print(f"Media (λ): {media_teorica:.4f}")
print(f"Desvío (√λ): {desvio_teorico:.4f}\n")

# 2. Parámetros Poblacionales (Nuestra cohorte simulada)
mu_poblacional = poblacion_infecciones.mean()
sigma_poblacional = poblacion_infecciones.std(ddof=0)

print(f"--- 2. PARÁMETROS POBLACIONALES (N={len(poblacion_infecciones)}) ---")
print(f"Media Poblacional (μ): {mu_poblacional:.4f}")
print(f"Desvío Poblacional (σ): {sigma_poblacional:.4f}\n")

# 3. Estadísticos Muestrales (Tomando un subgrupo al azar)
n_muestra = 50

print(f"--- 3. ESTADÍSTICOS MUESTRALES (n={n_muestra}) ---")
for i in range(1, 4):
    muestra_inf = poblacion_infecciones.sample(n=n_muestra, random_state=100+i)

    media_muestral = muestra_inf.mean()
    desvio_muestral = muestra_inf.std(ddof=1)

    print(f"Muestra {i} -> Media (x̄): {media_muestral:.4f} | Desvío (s): {desvio_muestral:.4f}")

--- 1. VALORES TEÓRICOS (Población infinita) ---
Media (λ): 2.0000
Desvío (√λ): 1.4142

--- 2. PARÁMETROS POBLACIONALES (N=10000000) ---
Media Poblacional (μ): 2.0003
Desvío Poblacional (σ): 1.4142

--- 3. ESTADÍSTICOS MUESTRALES (n=50) ---
Muestra 1 -> Media (x̄): 2.1000 | Desvío (s): 1.2976
Muestra 2 -> Media (x̄): 1.8600 | Desvío (s): 1.2456
Muestra 3 -> Media (x̄): 2.0800 | Desvío (s): 1.6517


> 💡 **Para pensar y debatir:**

1. **La propiedad fundamental de Poisson:** Elevá al cuadrado (multiplicá por sí mismo) el valor de cualquier Desvío (poblacional o muestral) que arrojó el código. El resultado se llama *Varianza*. ¿Qué relación notás entre ese número y la Media correspondiente? ¿Se cumple en la práctica la regla matemática de que la media y la varianza son iguales?
2. **El efecto del tamaño de la muestra ($n$):** Modificá la variable `n_muestra` en la celda anterior. Probá con valores muy chicos (ej: $n=5$) y valores más grandes (ej: $n=1000$). ¿Qué ocurre con la variabilidad de las medias muestrales ($\bar{x}$) obtenidas en las 3 extracciones a medida que el $n$ crece? ¿Se acercan o se alejan del parámetro poblacional ($\mu$) y del valor teórico ($\lambda$)?
3. **Población Finita vs. Infinita:** Notá que la media poblacional ($\mu$) calculada con los 100.000 datos es casi 2.0, pero tiene una ligerísima diferencia. Esto ocurre porque nuestra simulación computacional (aunque enorme) tiene un límite, mientras que el cálculo teórico asume infinitos eventos regidos perfectamente por las leyes de la probabilidad.

### Análisis de la Distribución Normal: Altura de la población

Finalmente, vamos a analizar la variable `Altura_cm`, generada a partir de una distribución Normal (la famosa campana de Gauss). Esta distribución es el estándar para modelar variables continuas en la naturaleza donde intervienen múltiples factores aditivos, como la estatura humana, el peso o la presión arterial.

Al construir la simulación, forzamos matemáticamente los parámetros teóricos: indicamos que el centro exacto de la campana (la media teórica $\mu$) fuera de 165.0 cm, con una dispersión (el desvío estándar teórico $\sigma$) de 12.0 cm.

En el siguiente bloque de código vamos a comparar:
1. Los parámetros **teóricos** que le pasamos al simulador.
2. Los parámetros poblacionales **reales** calculados sobre nuestra cohorte de 100.000 personas.
3. Los estadísticos **muestrales** calculados tomando subgrupos al azar.

In [ ]:
# Aislamos la variable continua de altura
poblacion_altura = df_poblacion['Altura_cm']

# 1. Valores Teóricos (Fijados en la simulación)
mu_teorica = 165.0          # parámetros repetidos de la simulación, conviene usar variables globales
sigma_teorica = 12.0

print("--- 1. VALORES TEÓRICOS (Input del simulador) ---")
print(f"Media (μ): {mu_teorica:.4f} cm")
print(f"Desvío (σ): {sigma_teorica:.4f} cm\n")

# 2. Parámetros Poblacionales (Nuestra cohorte simulada de 100k)
mu_poblacional = poblacion_altura.mean()
sigma_poblacional = poblacion_altura.std(ddof=0)

print(f"--- 2. PARÁMETROS POBLACIONALES (N={len(poblacion_altura)}) ---")
print(f"Media Poblacional (μ): {mu_poblacional:.4f} cm")
print(f"Desvío Poblacional (σ): {sigma_poblacional:.4f} cm\n")

# 3. Estadísticos Muestrales (Tomando subgrupos al azar)
n_muestra = 50

print(f"--- 3. ESTADÍSTICOS MUESTRALES (n={n_muestra}) ---")
for i in range(1, 4):
    muestra_alt = poblacion_altura.sample(n=n_muestra, random_state=200+i)

    media_muestral = muestra_alt.mean()
    desvio_muestral = muestra_alt.std(ddof=1) # Corrección de Bessel

    print(f"Muestra {i} -> Media (x̄): {media_muestral:.4f} | Desvío (s): {desvio_muestral:.4f}")

--- 1. VALORES TEÓRICOS (Input del simulador) ---
Media (μ): 165.0000 cm
Desvío (σ): 12.0000 cm

--- 2. PARÁMETROS POBLACIONALES (N=10000000) ---
Media Poblacional (μ): 165.0044 cm
Desvío Poblacional (σ): 11.9959 cm

--- 3. ESTADÍSTICOS MUESTRALES (n=50) ---
Muestra 1 -> Media (x̄): 164.3860 | Desvío (s): 14.1181
Muestra 2 -> Media (x̄): 164.4840 | Desvío (s): 12.2602
Muestra 3 -> Media (x̄): 163.6100 | Desvío (s): 12.6980


> 💡 **Para pensar y debatir:**

1. **La cercanía a la perfección:** Compará los valores teóricos con los poblacionales. Vas a notar que son extremadamente similares, pero no idénticos. ¿Por qué ocurre esta levísima diferencia incluso teniendo 100.000 datos?
2. **El tamaño de la muestra ($n$) y los estadísticos:** En la celda de código, cambiá el valor de `n_muestra` (probá con $n=10$, $n=100$, $n=5000$) y observá qué pasa con las medias muestrales ($\bar{x}$) y los desvíos muestrales ($s$).
   * A medida que $n$ aumenta, ¿las medias muestrales fluctúan menos o fluctúan más entre las distintas muestras?
   * Prestale especial atención al desvío muestral ($s$): al agrandar el $n$, ¿el valor de $s$ se hace cada vez más chiquito, o simplemente se estabiliza alrededor del valor teórico de 12.0? ¿Por qué el desvío de la muestra no tiende a cero, pero el error de nuestra estimación de la media sí?
3. **Implicancias prácticas:** Si tuvieras que estimar la altura promedio de toda la ciudad pero solo tenés presupuesto para medir a 50 personas, ¿te daría tranquilidad el resultado que obtuviste en la "Muestra 1"? ¿Qué información adicional necesitarías reportar junto a tu media muestral para ser riguroso científicamente?